# Multi-Decadal Glacier Retreat Time-Series Forecasting in the Ladakh Himalaya (2000–2024)
### A Comparative Deep Learning Benchmark: Random Forest vs. LSTM vs. TCN vs. DLinear

**Objective**:
Couples satellite-derived glacier area changes (Landsat 7/8/9 & Sentinel-2) with ERA5-Land climate forcing (summer temperature, winter snowfall, solar radiation) and topography (Copernicus DEM) to forecast annual glacier retreat across glaciers in Ladakh (RGI Region 14).

**Methodology Synthesis**:
- **Spatial Mapper**: *Sahu & Yasmin (2026)* — Multi-spectral semantic boundary mapping.
- **Climate-Driven Forecaster**: *Van der Meer et al. (2026)* & *GlacierCastAI (2024)* — Long-term sequence modeling under atmospheric forcing.

## 1. System Setup & Compute Check

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device:  {device}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")

## 2. Load Dataset: Google Drive GEE Export (`MyDrive/time_series`) or Fallback Generator
Automatically mounts Google Drive to check for the real GEE export: `MyDrive/time_series/ladakh_glaciers_2000_2024_real.csv`.
If not yet exported, falls back to the physically grounded sample dataset so training can proceed immediately.

In [ ]:
# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_CSV_PATH = '/content/drive/MyDrive/time_series/ladakh_glaciers_2000_2024_real.csv'
LOCAL_CSV_PATH = 'data/ladakh_glaciers_2000_2024.csv'

def generate_ladakh_dataset(num_glaciers=150, start_year=2000, end_year=2024):
    np.random.seed(42)
    years = np.arange(start_year, end_year + 1)
    records = []
    for gid in range(1, num_glaciers + 1):
        glacier_code = f"RGI60-14.{gid:05d}"
        elevation = np.random.uniform(4900, 6200)
        slope = np.random.uniform(12.0, 28.0)
        initial_area = np.clip(np.random.lognormal(mean=2.0, sigma=0.8), 1.2, 45.0)
        base_summer_temp = 7.5 - (elevation - 5000) * 0.0065
        base_winter_precip = 180.0 + (elevation - 5000) * 0.05
        base_solar = 245.0
        current_area = initial_area
        for i, y in enumerate(years):
            summer_temp = base_summer_temp + 0.038 * i + np.random.normal(0, 0.65)
            precip_anomaly = 25.0 * np.sin(2 * np.pi * i / 7.0) + np.random.normal(0, 30.0)
            winter_precip = np.clip(base_winter_precip + precip_anomaly, 60.0, 420.0)
            solar_rad = base_solar - 0.15 * (winter_precip - base_winter_precip) + np.random.normal(0, 8.0)
            temp_excess = max(0.0, summer_temp - 1.5)
            melt_factor = 0.022 * (temp_excess ** 1.3)
            accum_factor = 0.00008 * winter_precip
            solar_melt = 0.00012 * (solar_rad - 220.0)
            delta_fraction = - (melt_factor + solar_melt - accum_factor) * (1.0 + 0.02 * (30.0 - slope) / 10.0)
            delta_fraction = np.clip(delta_fraction, -0.045, 0.008)
            current_area = max(0.2, current_area * (1.0 + delta_fraction))
            ndsi = np.clip(0.82 - 0.025 * temp_excess + 0.0003 * winter_precip + np.random.normal(0, 0.025), 0.35, 0.95)
            records.append({
                "glacier_id": glacier_code, "year": int(y),
                "glacier_area_km2": round(float(current_area), 4),
                "ndsi_mean": round(float(ndsi), 4),
                "temp_summer_c": round(float(summer_temp), 2),
                "precip_winter_mm": round(float(winter_precip), 2),
                "solar_radiation_wm2": round(float(solar_rad), 2),
                "elevation_mean_m": round(float(elevation), 1),
                "slope_mean_deg": round(float(slope), 2)
            })
    return pd.DataFrame(records)

if os.path.exists(DRIVE_CSV_PATH):
    print(f"-> SUCCESS: Found REAL GEE satellite & climate dataset in Google Drive: {DRIVE_CSV_PATH}")
    df = pd.read_csv(DRIVE_CSV_PATH)
    # Remove GEE metadata columns if present
    for col in ['system:index', '.geo']:
        if col in df.columns:
            df = df.drop(columns=[col])
elif os.path.exists(LOCAL_CSV_PATH):
    print(f"-> Using pre-compiled dataset from local repository: {LOCAL_CSV_PATH}")
    df = pd.read_csv(LOCAL_CSV_PATH)
else:
    print("-> Note: Real GEE dataset not yet detected in MyDrive/time_series/.")
    print("-> Generating physically-consistent sample dataset for Ladakh (150 glaciers)...")
    df = generate_ladakh_dataset(num_glaciers=150)

# Ensure correct data types
df['year'] = df['year'].astype(int)
numeric_cols = ['glacier_area_km2', 'ndsi_mean', 'temp_summer_c', 'precip_winter_mm', 'solar_radiation_wm2', 'elevation_mean_m', 'slope_mean_deg']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna().sort_values(['glacier_id', 'year']).reset_index(drop=True)

print(f"Dataset ready: {len(df)} total annual records across {df['glacier_id'].nunique()} glaciers (2000-2024).")
df.head()

## 3. Chronological Train-Val-Test Split (Zero Data Leakage)
- **Train Set**: Windows $\le 2017$ (18 historical years)
- **Validation Set**: Windows $2018 - 2020$ (3 years)
- **Test Set**: Windows $2021 - 2024$ (4 evaluation years)

In [ ]:
feature_cols = ['glacier_area_km2', 'ndsi_mean', 'temp_summer_c', 'precip_winter_mm', 'solar_radiation_wm2', 'elevation_mean_m', 'slope_mean_deg']
target_col = 'glacier_area_km2'
LOOKBACK = 5
HORIZON = 1

train_X, train_y, val_X, val_y, test_X, test_y = [], [], [], [], [], []
for gid in df['glacier_id'].unique():
    gdf = df[df['glacier_id'] == gid].sort_values('year').reset_index(drop=True)
    years = gdf['year'].values
    mat = gdf[feature_cols].values
    target = gdf[target_col].values
    for i in range(len(years) - LOOKBACK - HORIZON + 1):
        wx = mat[i : i + LOOKBACK]
        wy = target[i + LOOKBACK + HORIZON - 1]
        pyr = years[i + LOOKBACK + HORIZON - 1]
        if pyr <= 2017:
            train_X.append(wx); train_y.append(wy)
        elif pyr <= 2020:
            val_X.append(wx); val_y.append(wy)
        else:
            test_X.append(wx); test_y.append(wy)

train_X, train_y = np.array(train_X, dtype=np.float32), np.array(train_y, dtype=np.float32).reshape(-1, 1)
val_X, val_y = np.array(val_X, dtype=np.float32), np.array(val_y, dtype=np.float32).reshape(-1, 1)
test_X, test_y = np.array(test_X, dtype=np.float32), np.array(test_y, dtype=np.float32).reshape(-1, 1)

scaler = StandardScaler().fit(train_X.reshape(-1, len(feature_cols)))
train_X_s = scaler.transform(train_X.reshape(-1, len(feature_cols))).reshape(train_X.shape)
val_X_s = scaler.transform(val_X.reshape(-1, len(feature_cols))).reshape(val_X.shape)
test_X_s = scaler.transform(test_X.reshape(-1, len(feature_cols))).reshape(test_X.shape)

target_scaler = StandardScaler().fit(train_y)
train_y_s = target_scaler.transform(train_y)
val_y_s = target_scaler.transform(val_y)
test_y_s = target_scaler.transform(test_y)

class SeqDataset(Dataset):
    def __init__(self, X, y): self.X = torch.from_numpy(X).float(); self.y = torch.from_numpy(y).float()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(train_X_s, train_y_s), batch_size=32, shuffle=True)
val_loader = DataLoader(SeqDataset(val_X_s, val_y_s), batch_size=32, shuffle=False)
test_loader = DataLoader(SeqDataset(test_X_s, test_y_s), batch_size=32, shuffle=False)

print(f"Windows built: {len(train_X)} train, {len(val_X)} val, {len(test_X)} test.")

## 4. Model Architectures Implementation
1. **Random Forest** (Classical non-deep learning baseline)
2. **LSTM** (Recurrent sequence memory with Forget/Input/Output gates)
3. **TCN** (1D Dilated Causal Convolutional Network)
4. **DLinear** (Trend-Seasonal Decomposition Model - AAAI 2023)

In [ ]:
# Model 2: LSTM
class GlacierLSTM(nn.Module):
    def __init__(self, in_dim=7, hidden=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

# Model 3: TCN (Temporal Convolutional Network)
class Chomp1d(nn.Module):
    def __init__(self, s): super().__init__(); self.s = s
    def forward(self, x): return x[:, :, :-self.s].contiguous() if self.s > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, n_in, n_out, k=2, d=1):
        super().__init__()
        pad = (k - 1) * d
        self.net = nn.Sequential(
            nn.Conv1d(n_in, n_out, k, padding=pad, dilation=d), Chomp1d(pad), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(n_out, n_out, k, padding=pad, dilation=d), Chomp1d(pad), nn.ReLU(), nn.Dropout(0.2)
        )
        self.down = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.down is None else self.down(x)
        return self.relu(self.net(x) + res)

class GlacierTCN(nn.Module):
    def __init__(self, in_dim=7, channels=[32, 64], k=2):
        super().__init__()
        layers = []
        for i, ch in enumerate(channels):
            layers.append(TemporalBlock(in_dim if i==0 else channels[i-1], ch, k=k, d=2**i))
        self.net = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], 1)
    def forward(self, x):
        y = self.net(x.transpose(1, 2))
        return self.fc(y[:, :, -1])

# Model 4: DLinear (AAAI 2023)
class GlacierDLinear(nn.Module):
    def __init__(self, lookback=5, in_dim=7, k=3):
        super().__init__()
        self.k = k
        self.avg = nn.AvgPool1d(kernel_size=k, stride=1, padding=0)
        self.l_seasonal = nn.Linear(lookback * in_dim, 1)
        self.l_trend = nn.Linear(lookback * in_dim, 1)
    def forward(self, x):
        xt = x.transpose(1, 2)
        pl, pr = (self.k - 1) // 2, (self.k - 1) - (self.k - 1) // 2
        xp = torch.cat([xt[:, :, 0:1].repeat(1, 1, pl), xt, xt[:, :, -1:].repeat(1, 1, pr)], dim=-1)
        trend = self.avg(xp).transpose(1, 2)
        seasonal = x - trend
        return self.l_seasonal(seasonal.reshape(x.shape[0], -1)) + self.l_trend(trend.reshape(x.shape[0], -1))

print("All 4 Model Architectures defined successfully!")

## 5. Comparative Benchmark Training Loop

In [ ]:
def train_pt(model, epochs=40, lr=0.002):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.MSELoss()
    best_loss, best_w = float('inf'), None
    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval(); vloss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vloss += crit(model(xb), yb).item() * len(xb)
        vloss /= len(val_loader.dataset)
        if vloss < best_loss:
            best_loss = vloss; best_w = model.state_dict().copy()
    model.load_state_dict(best_w)
    return model

y_true_km2 = test_y.ravel()
results = []

# 1. Random Forest
print("Training Random Forest (Baseline)...")
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(train_X_s.reshape(len(train_X), -1), train_y.ravel())
rf_preds = rf.predict(test_X_s.reshape(len(test_X), -1))
t_rf = round(time.time() - t0, 2)
results.append({"Model": "Random Forest", "R2": r2_score(y_true_km2, rf_preds), "RMSE": np.sqrt(mean_squared_error(y_true_km2, rf_preds)), "MAE": mean_absolute_error(y_true_km2, rf_preds), "Time_s": t_rf})

# 2. LSTM
print("Training LSTM (Recurrent Network)...")
t0 = time.time(); lstm = train_pt(GlacierLSTM(in_dim=7, hidden=64))
lstm.eval()
with torch.no_grad():
    lstm_preds = target_scaler.inverse_transform(lstm(torch.from_numpy(test_X_s).to(device)).cpu().numpy()).ravel()
t_lstm = round(time.time() - t0, 2)
results.append({"Model": "LSTM", "R2": r2_score(y_true_km2, lstm_preds), "RMSE": np.sqrt(mean_squared_error(y_true_km2, lstm_preds)), "MAE": mean_absolute_error(y_true_km2, lstm_preds), "Time_s": t_lstm})

# 3. TCN
print("Training TCN (1D Dilated Conv)...")
t0 = time.time(); tcn = train_pt(GlacierTCN(in_dim=7))
tcn.eval()
with torch.no_grad():
    tcn_preds = target_scaler.inverse_transform(tcn(torch.from_numpy(test_X_s).to(device)).cpu().numpy()).ravel()
t_tcn = round(time.time() - t0, 2)
results.append({"Model": "TCN", "R2": r2_score(y_true_km2, tcn_preds), "RMSE": np.sqrt(mean_squared_error(y_true_km2, tcn_preds)), "MAE": mean_absolute_error(y_true_km2, tcn_preds), "Time_s": t_tcn})

# 4. DLinear
print("Training DLinear (Trend-Seasonal Decomposition)...")
t0 = time.time(); dlinear = train_pt(GlacierDLinear(lookback=5, in_dim=7))
dlinear.eval()
with torch.no_grad():
    dlinear_preds = target_scaler.inverse_transform(dlinear(torch.from_numpy(test_X_s).to(device)).cpu().numpy()).ravel()
t_dlin = round(time.time() - t0, 2)
results.append({"Model": "DLinear", "R2": r2_score(y_true_km2, dlinear_preds), "RMSE": np.sqrt(mean_squared_error(y_true_km2, dlinear_preds)), "MAE": mean_absolute_error(y_true_km2, dlinear_preds), "Time_s": t_dlin})

res_df = pd.DataFrame(results).round(4)
res_df

## 6. Publication-Quality Visualizations & Forecast Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Model Comparison
sns.barplot(data=res_df, x='Model', y='R2', hue='Model', ax=axes[0], palette='Blues_d', legend=False)
axes[0].set_title("Forecasting Accuracy (R² Score)", fontsize=13, weight='bold')
axes[0].set_ylim(min(0.90, res_df['R2'].min() - 0.02), 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10)

# Plot 2: Sample Glacier Test Retreat Prediction (2021-2024)
sample_idx = slice(0, 4)
eval_years = [2021, 2022, 2023, 2024]
axes[1].plot(eval_years, y_true_km2[sample_idx], 'k-o', label='Actual Area (km²)', linewidth=2.5)
axes[1].plot(eval_years, rf_preds[sample_idx], '--s', label='Random Forest', alpha=0.8)
axes[1].plot(eval_years, lstm_preds[sample_idx], '--^', label='LSTM', alpha=0.8)
axes[1].plot(eval_years, tcn_preds[sample_idx], '--d', label='TCN', alpha=0.8)
axes[1].plot(eval_years, dlinear_preds[sample_idx], '--*', label='DLinear', linewidth=2)
axes[1].set_title("Glacier Retreat Trajectory (Test Period: 2021–2024)", fontsize=13, weight='bold')
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Glacier Surface Area (km²)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()